In [2]:
# Experiment 11: Sampling sweep over temperature, top-k and top-p on the golden set; classify failures
!pip -q install pymongo
import time, random
import pandas as pd
from google.colab import userdata
from google import genai
from google.genai import types
from pymongo import MongoClient

MODEL = "gemini-3.1-flash-lite"
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"), http_options={"timeout": 30000})
db = MongoClient(userdata.get("MONGODB_URI"), serverSelectionTimeoutMS=8000)["genai_course"]
golden = list(db.golden_set.find({}, {"_id": 0}))
random.seed(7); subset = random.sample(golden, 8)          # 8 questions x 5 configs = 40 calls (free-tier friendly)
print("Loaded golden set:", len(golden), "| sweeping on", len(subset), "questions")

CONFIGS = {
    "greedy T=0":            dict(temperature=0.0),
    "T=0.7, top_p=0.9":      dict(temperature=0.7, top_p=0.9),
    "T=1.0, top_k=40":       dict(temperature=1.0, top_k=40),
    "T=1.5, top_p=1.0":      dict(temperature=1.5, top_p=1.0),
    "T=2.0, top_k=100":      dict(temperature=2.0, top_k=100),
}

def ask(prompt, cfg, retries=4):
    for a in range(retries):
        try:
            return client.models.generate_content(model=MODEL, contents=prompt,
                       config=types.GenerateContentConfig(max_output_tokens=200, **cfg))
        except Exception as e:
            print("retry", a + 1, str(e)[:50]); time.sleep(5 * (a + 1))
    return None

def classify(ans, keywords, out_tokens):
    a = (ans or "").lower().replace("-", " ")
    if not a.strip():                                             return "empty"
    if all(k.lower() in a for k in keywords):                    return "correct"
    if any(s in a for s in ["i don't", "i do not", "cannot", "please provide", "not sure"]): return "refusal / unknown"
    if out_tokens and out_tokens >= 190:                          return "verbose / truncated"
    return "incorrect / hallucinated"

rows = []
for name, cfg in CONFIGS.items():
    for g in subset:
        t = time.time()
        r = ask("Answer in one or two sentences: " + g["question"], cfg)
        ans = (r.text if r and r.text else "")
        out_t = r.usage_metadata.candidates_token_count if r else 0
        rows.append({"config": name, **cfg, "id": g["id"], "label": classify(ans, g["expected_keywords"], out_t),
                     "latency_s": round(time.time() - t, 2), "output_tokens": out_t, "answer": ans[:70].replace("\n", " ")})
        time.sleep(4)

df = pd.DataFrame(rows)
db.sampling_sweep.delete_many({}); db.sampling_sweep.insert_many(df.drop(columns=["answer"]).to_dict("records"))
summary = df.groupby("config").agg(accuracy=("label", lambda s: round((s == "correct").mean(), 2)),
                                   avg_latency_s=("latency_s", "mean"), avg_output_tokens=("output_tokens", "mean")).round(2)
print("\nSweep summary:"); print(summary.sort_values("accuracy", ascending=False).to_string())
print("\nFailure classification sheet (counts per config):")
print(pd.crosstab(df.config, df.label).to_string())
print("\nExamples of failures:")
print(df[df.label != "correct"][["config", "id", "label", "answer"]].head(10).to_string(index=False))

Loaded golden set: 32 | sweeping on 8 questions
retry 1 503 UNAVAILABLE. {'error': {'code': 503, 'message'
retry 2 503 UNAVAILABLE. {'error': {'code': 503, 'message'
retry 3 504 DEADLINE_EXCEEDED. {'error': {'code': 504, 'me
retry 4 503 UNAVAILABLE. {'error': {'code': 503, 'message'
retry 1 503 UNAVAILABLE. {'error': {'code': 503, 'message'
retry 1 504 DEADLINE_EXCEEDED. {'error': {'code': 504, 'me
retry 1 503 UNAVAILABLE. {'error': {'code': 503, 'message'
retry 1 504 DEADLINE_EXCEEDED. {'error': {'code': 504, 'me
retry 1 503 UNAVAILABLE. {'error': {'code': 503, 'message'
retry 2 503 UNAVAILABLE. {'error': {'code': 503, 'message'
retry 3 503 UNAVAILABLE. {'error': {'code': 503, 'message'
retry 1 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'm
retry 1 503 UNAVAILABLE. {'error': {'code': 503, 'message'
retry 1 503 UNAVAILABLE. {'error': {'code': 503, 'message'
retry 1 504 DEADLINE_EXCEEDED. {'error': {'code': 504, 'me
retry 1 503 UNAVAILABLE. {'error': {'code': 503, 'message'
retry 1 